In [ ]:
!pip install transformers datasets evaluate bert-score accelerate bitsandbytes rouge_score


In [ ]:
import os
import logging
import re
import glob
import numpy as np
import torch
import gc
from pickle import UnpicklingError

from datasets import Dataset, disable_caching
from transformers import (
    LEDForConditionalGeneration,
    LEDTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from evaluate import load
from bert_score import score as bert_score

def find_latest(res_dir, err_dir):
    """
    1) Scan RESULTS_DIR for HuggingFace-style checkpoints and pick the highest step.
    2) If none found, but ERROR_DIR contains model weights, return ERROR_DIR.
    3) Otherwise return None (train from scratch).
    """
    latest_ckpt, max_step = None, -1

    # 1) Look for checkpoints like RESULTS_DIR/checkpoint-*
    for ck in glob.glob(os.path.join(res_dir, "checkpoint-*")):
        m = re.search(r"checkpoint-(\d+)", ck)
        if not m:
            continue
        step = int(m.group(1))
        if step > max_step and any(
            os.path.exists(os.path.join(ck, fname))
            for fname in ["pytorch_model.bin", "model.safetensors"]
        ):
            max_step, latest_ckpt = step, ck

    # 2) If we found one, use it
    if latest_ckpt is not None:
        return latest_ckpt

    # 3) Fallback to ERROR_DIR if it has weights
    if os.path.isdir(err_dir) and any(
        os.path.exists(os.path.join(err_dir, fname))
        for fname in ["pytorch_model.bin", "model.safetensors"]
    ):
        return err_dir

    # 4) Nothing to resume from
    return None

class LegalDataset:
    def __init__(self, split, base_path, data_type, max_docs):
        self.root = os.path.join(base_path, "dataset", data_type)
        splits = {"train":"train-data", "validation":"validation-data", "test":"test-data"}
        self.dir = os.path.join(self.root, splits[split])
        self.jud = os.path.join(self.dir, "judgement")
        self.sum = os.path.join(self.dir, "summary")
        if not os.path.isdir(self.dir):
            raise FileNotFoundError(f"Split not found: {self.dir}")
        self.max_docs = max_docs

    def load(self):
        docs, sums = {}, {}
        count = 0
        for p in glob.iglob(os.path.join(self.jud, "*.txt")):
            if count >= self.max_docs:
                break
            try:
                with open(p, 'r', encoding='utf-8', errors='ignore') as f:
                    docs[os.path.basename(p)] = f.read()
                count += 1
            except:
                continue

        count = 0
        for p in glob.iglob(os.path.join(self.sum, "*.txt")):
            if count >= self.max_docs:
                break
            try:
                with open(p, 'r', encoding='utf-8', errors='ignore') as f:
                    sums[os.path.basename(p)] = f.read()
                count += 1
            except:
                continue

        ids, doc_list, sum_list = [], [], []
        for fn, text in docs.items():
            if fn in sums and text.strip() and sums[fn].strip():
                ids.append(fn)
                doc_list.append(text[:20000] + '...' if len(text) > 20000 else text)
                sum_list.append(sums[fn])
        return Dataset.from_dict({"id": ids, "document": doc_list, "summary": sum_list})

def tokenize_fn(examples, tokenizer, max_in_len, max_tgt_len):
    in_ids, at_masks, lbls = [], [], []
    for doc, summ in zip(examples["document"], examples["summary"]):
        enc = tokenizer(doc, padding='max_length', truncation=True,
                        max_length=max_in_len)
        dec = tokenizer(text_target=summ, padding='max_length',
                        truncation=True, max_length=max_tgt_len)
        in_ids.append(enc.input_ids)
        at_masks.append(enc.attention_mask)
        lbls.append(dec.input_ids)
    return {"input_ids": in_ids,
            "attention_mask": at_masks,
            "labels": lbls}

def compute_metrics_factory(tokenizer):
    rouge = load("rouge")

    def preprocess_logits_for_metrics(logits, labels):
        if isinstance(logits, tuple):
            logits = logits[0]
        return logits.argmax(dim=-1)

    def compute_metrics(eval_pred):
        preds, labels = eval_pred
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels = np.where(labels == -100, tokenizer.pad_token_id, labels)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        results = rouge.compute(predictions=decoded_preds,
                                references=decoded_labels,
                                use_stemmer=True)
        return {k: round(v * 100, 4) for k, v in results.items()}

    return preprocess_logits_for_metrics, compute_metrics

def main():
    disable_caching()
    logging.basicConfig(level=logging.INFO,
                        format='%(asctime)s - %(levelname)s - %(message)s',
                        force=True)
    logger = logging.getLogger(__name__)

    # Paths & Config
    BASE        = "/kaggle/input/legal-cases-processed"
    OUT         = "/kaggle/working"
    RESULTS_DIR = os.path.join(OUT, "results")
    LOGS_DIR    = os.path.join(OUT, "logs")
    FINAL_DIR   = os.path.join(OUT, "led_finetuned_final")
    ERROR_DIR   = os.path.join(OUT, "model_on_error")
    CACHE_DIR   = os.path.join(OUT, "local_model_cache")

    MODEL_NAME  = "Dharil/legal-led-base-16384-finetuned-legal-tech-IN"
    MAX_IN_LEN  = 4096
    MAX_TGT_LEN = 1024
    MAX_DOCS    = 1300
    MAX_EVAL    = 250
    DATA_TYPE   = "IN-Abs"

    for d in [RESULTS_DIR, LOGS_DIR, FINAL_DIR, ERROR_DIR, CACHE_DIR]:
        os.makedirs(d, exist_ok=True)

    # Load and preprocess datasets
    train_ds = LegalDataset("train", BASE, DATA_TYPE, MAX_DOCS).load()
    val_ds   = LegalDataset("validation", BASE, DATA_TYPE, 1035).load()
    test_ds  = LegalDataset("test", BASE, DATA_TYPE, 100).load()

    logger.info(f"Dataset sizes -> Train: {len(train_ds)}, "
                f"Val: {len(val_ds)}, Test: {len(test_ds)}")

    tokenizer = LEDTokenizer.from_pretrained(MODEL_NAME)
    model     = LEDForConditionalGeneration.from_pretrained(MODEL_NAME)
    tokenizer.save_pretrained(CACHE_DIR)
    model.save_pretrained(CACHE_DIR)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    train_ds = train_ds.map(
        lambda ex: tokenize_fn(ex, tokenizer, MAX_IN_LEN, MAX_TGT_LEN),
        batched=True, batch_size=4, remove_columns=train_ds.column_names
    )
    val_ds = val_ds.map(
        lambda ex: tokenize_fn(ex, tokenizer, MAX_IN_LEN, MAX_TGT_LEN),
        batched=True, batch_size=4, remove_columns=val_ds.column_names
    )
    test_ds = test_ds.map(
        lambda ex: tokenize_fn(ex, tokenizer, MAX_IN_LEN, MAX_TGT_LEN),
        batched=True, batch_size=4, remove_columns=test_ds.column_names
    )

    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

    # Limit validation size
    if len(val_ds) > MAX_EVAL:
        idxs = np.random.choice(len(val_ds), MAX_EVAL, replace=False)
        val_ds = val_ds.select(idxs)

    preprocess_logits_for_metrics, compute_metrics = compute_metrics_factory(tokenizer)

    # Training arguments
    training_args = TrainingArguments(
        output_dir=RESULTS_DIR,
        logging_dir=LOGS_DIR,
        report_to="none",
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_rouge2",
        greater_is_better=True,
        learning_rate=5e-4,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,
        gradient_checkpointing=True,
        fp16=True,
        optim="adafactor",
        eval_accumulation_steps=1
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        label_pad_token_id=tokenizer.pad_token_id
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        preprocess_logits_for_metrics=preprocess_logits_for_metrics
    )

    # --- TRAINING ---
    ckpt = find_latest(RESULTS_DIR, ERROR_DIR)
    try:
        trainer.train(resume_from_checkpoint=ckpt)
    except (UnpicklingError, RuntimeError) as e:
        logger.warning(f"Could not resume from {ckpt!r} ({e}), restarting from scratch.")
        trainer.train(resume_from_checkpoint=None)

    trainer.save_model(FINAL_DIR)
    tokenizer.save_pretrained(FINAL_DIR)

    # --- FINAL EVALUATION ---
    logger.info("Running generation-based evaluation on test set...")
    preds_output = trainer.predict(test_ds, metric_key_prefix="test")
    gen_ids = preds_output.predictions
    if isinstance(gen_ids, tuple):
        gen_ids = gen_ids[0]
    preds = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
    refs  = LegalDataset("test", BASE, DATA_TYPE, MAX_DOCS).load()["summary"]

    rouge = load("rouge")
    rouge_scores = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    logger.info("Test ROUGE: " + ", ".join(f"{k}: {v*100:.2f}" for k,v in rouge_scores.items()))

    P, R, F1 = bert_score(preds, refs, lang="en", batch_size=4,
                           device="cpu", rescale_with_baseline=True)
    logger.info(f"BERTScore F1 (mean): {float(F1.mean().item()):.4f}")

    # Cleanup
    del model, tokenizer, trainer
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

if __name__ == "__main__":
    main()


In [ ]:
from transformers import LEDTokenizer, LEDForConditionalGeneration
import torch

# Sample legal judgement (India region, court document)
sample_judgment = """                                     IN THE HIGH COURT OF DELHI AT NEW DELHI

                                               CRL.M.C. No. 1234/2025

                                               Date of Decision: April 20, 2025

[Applicant Name]
Rajiv Sharma
                                                                                                     ... Petitioner

                                                          Versus

[Respondent Name]
State of NCT of Delhi & Anr.
                                                                                                   ... Respondent

CORAM:
HON'BLE MR. JUSTICE Anup Kumar Mendiratta

                                                      JUDGMENT

[Date of Hearing]
April 15, 2025

1.  This petition under Section 482 of the Code of Criminal Procedure, 1973 (CrPC) has been filed by the petitioner, Rajiv Sharma, seeking to quash the First Information Report (FIR) No. 567/2024, registered at Police Station Defence Colony, under Section 420 of the Indian Penal Code, 1860 (IPC), and all consequential proceedings arising therefrom.

2.  The brief facts of the case, as alleged in the FIR, are that the respondent No. 2, Mr. Vikas Gupta, lodged a complaint alleging that the petitioner cheated him of a sum of ₹10,00,000/- in a property transaction. The FIR was registered based on this complaint.

3.  The petitioner's case, in brief, is that the allegations are false and frivolous, and the dispute is purely civil in nature arising out of a contractual agreement. The petitioner has annexed certain documents in support of their contentions, including copies of the agreement to sell and bank statements.

4.  The respondent, through the State, has opposed the petition, contending that there are serious allegations of cheating against the petitioner which require thorough investigation. The learned Public Prosecutor submitted that the FIR discloses the commission of a cognizable offence and that the investigation is at a preliminary stage.

5.  I have heard the learned counsel for the petitioner, Mr. S.K. Singh, and the learned APP for the State, Ms. Meenakshi Chauhan, and have perused the material on record.

6.  The primary question before this Court is whether the allegations in the FIR, even if taken at their face value, constitute a cognizable offence warranting further investigation, or whether the proceedings should be quashed.

7.  In the present case, the allegations in the FIR primarily relate to a breach of the terms of the agreement to sell. While the complainant alleges that the petitioner had dishonest intentions from the inception, the documents on record, particularly the agreement to sell, suggest a contractual relationship between the parties. The line between a mere breach of contract and the offence of cheating is often thin and depends on the existence of dishonest intention at the time of making the promise. In the absence of clear and cogent material indicating such dishonest intention from the outset, allowing the criminal proceedings to continue would be an abuse of the process of law. Reliance is placed on the judgment of the Hon'ble Supreme Court in *Gian Singh v. State of Punjab & Anr.*, (2012) 10 SCC 303, which emphasizes that criminal proceedings are not a shortcut for settling civil disputes.

8.  In light of the above discussion and analysis, this Court is of the considered opinion that the allegations in the FIR, even if taken at their face value, do not prima facie disclose the commission of an offence of cheating under Section 420 IPC. The matter appears to be essentially civil in nature. Therefore, to allow the criminal proceedings to continue would be unwarranted.

9.  Therefore, in the result, the petition is allowed. The FIR No. 567/2024, registered at Police Station Defence Colony, under Section 420 of the Indian Penal Code, 1860, and all consequential proceedings arising therefrom are hereby quashed.

                                                                                                     Anup Kumar Mendiratta
                                                                                                         JUDGE

April 20, 2025
"""

# Load model and tokenizer from your checkpoint directory
MODEL_DIR = "/kaggle/working/led_finetuned_final"
tokenizer = LEDTokenizer.from_pretrained(MODEL_DIR)
model = LEDForConditionalGeneration.from_pretrained(MODEL_DIR)

# Move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Sanity check
print("Model device:", next(model.parameters()).device)

# Tokenize and generate
inputs = tokenizer(
    sample_judgment,
    max_length=3072,
    return_tensors="pt",
    truncation=True
)

# Move inputs to GPU
inputs = {k: v.to(device) for k, v in inputs.items()}
print("Input IDs device:", inputs["input_ids"].device)

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=2048,
        num_beams=5,
        early_stopping=False
    )

# Decode and print the output
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n--- Generated Summary ---\n")
print(generated_text)
